# Recursive 10-Minute Rootzone Model`n`nNotebook version of the stabilized recursive EC/pH workflow with gap-bin calibration.`n

In [ ]:
from __future__ import annotations

import argparse
import json
import math
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

try:
    import xgboost as xgb
except Exception:
    xgb = None

ACID_FERTS = ["Phosphoric acid[mg]-H3PO4"]
SALT_FERTS = [
    "Monopotassium Phosphate[mg] -KH2PO4",
    "Potassium Chloride[mg] - KCL",
    "Kortin [mg]",
    "Ammonium Nitrate [mg] -NH4NO3",
    "Gypsum - CaSO4*2H2O [mg]",
]
PH_FEATURES = [
    "ph_current", "ec_current", "step_hours", "hours_since_anchor",
    "irrigation_step", "fert_total_step", "acid_step", "ET0_step",
    "temp_step", "soil_temp_step", "radiation_step", "canopy_step",
    "fert_conc_step", "acid_conc_step", "vpd_step", "transpiration_step",
    "evapo_step", "hist_irr_recent", "hist_irr_mid", "hist_irr_prior",
    "hist_acid_decay", "hist_salt_buildup", "hist_hrs_since_fert",
    "hist_hrs_since_irr", "recent_temp_mean_6h", "recent_rad_integral_6h",
    "recent_vpd_mean_6h", "hour_sin", "hour_cos",
]
EC_FEATURES = [
    "ec_current", "ph_current", "step_hours", "hours_since_anchor",
    "irrigation_step", "fert_total_step", "salt_step", "ET0_step",
    "temp_step", "soil_temp_step", "canopy_step", "fert_conc_step",
    "salt_balance_step", "log_ec_drive_step", "transpiration_step",
    "evapo_step", "hist_irr_recent", "hist_irr_mid", "hist_irr_prior",
    "hist_salt_ramp", "hist_salt_buildup", "hist_hrs_since_fert",
    "hist_hrs_since_irr", "recent_et0_sum_6h", "recent_temp_mean_6h",
    "recent_vpd_mean_6h", "hour_sin", "hour_cos",
]
XGB_PARAMS = {
    "ph": dict(n_estimators=500, learning_rate=0.05, max_depth=4, min_child_weight=8,
               subsample=0.8, colsample_bytree=0.7, reg_alpha=2.0, reg_lambda=2.0,
               random_state=42, n_jobs=-1, tree_method="hist"),
    "ec": dict(n_estimators=300, learning_rate=0.05, max_depth=3, min_child_weight=12,
               subsample=0.85, colsample_bytree=0.7, reg_alpha=1.0, reg_lambda=2.0,
               random_state=42, n_jobs=-1, tree_method="hist"),
}
RF_PARAMS = {
    "ph": dict(n_estimators=500, max_depth=12, min_samples_leaf=8, max_features=0.6, random_state=42, n_jobs=-1),
    "ec": dict(n_estimators=500, max_depth=10, min_samples_leaf=10, max_features=0.6, random_state=42, n_jobs=-1),
}

@dataclass
class RunConfig:
    input_path: str
    output_dir: str
    step_minutes: int = 10
    warmup_segments: int = 50
    model_family: str = "auto"
    pseudo_label_mode: str = "driver_blend"
    pseudo_uniform_blend: float = 0.60
    max_training_gap_hours: float = 72.0
    long_gap_downweight_hours: float = 24.0
    reference_segment_hours: float = 6.0
    ph_min: float = 0.0
    ph_max: float = 14.0

@dataclass
class Segment:
    segment_id: int
    start_time: pd.Timestamp
    end_time: pd.Timestamp
    gap_hours: float
    ph_start: float
    ph_end: float
    ec_start: float
    ec_end: float


def to_num(series, default=0.0):
    return pd.to_numeric(series, errors="coerce").fillna(default) if series is not None else pd.Series(dtype=float)

def slice_df(df, start, end):
    return df.loc[(df.index >= start) & (df.index < end)] if start < end else df.iloc[0:0]

def sum_avail(df, cols):
    cols = [c for c in cols if c in df.columns]
    return float(df[cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).sum().sum()) if cols and len(df) else 0.0

def sum_series(df, cols):
    cols = [c for c in cols if c in df.columns]
    return df[cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).sum(axis=1) if cols and len(df) else pd.Series(0.0, index=df.index)

def fert_any(df):
    if len(df) == 0:
        return pd.Series(dtype=float)
    if "fertilization_flag" in df.columns:
        return to_num(df["fertilization_flag"])
    a = to_num(df["fertilization_type_a_flag"]) if "fertilization_type_a_flag" in df.columns else pd.Series(0.0, index=df.index)
    b = to_num(df["fertilization_type_b_flag"]) if "fertilization_type_b_flag" in df.columns else pd.Series(0.0, index=df.index)
    return ((a > 0) | (b > 0)).astype(float)

def vpd(temp_c, rh_pct):
    t = np.asarray(temp_c, dtype=float)
    rh = np.asarray(rh_pct, dtype=float)
    es = 0.6108 * np.exp((17.27 * t) / (t + 237.3))
    return es * (1.0 - np.clip(rh, 0.0, 100.0) / 100.0)

def hrs_since_last_positive(series, current_time, default_hours):
    if len(series) == 0:
        return default_hours
    hits = series[series > 0]
    return float(max(0.0, (current_time - hits.index.max()).total_seconds() / 3600.0)) if len(hits) else default_hours

def normalize(values):
    arr = np.nan_to_num(np.asarray(values, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    mean = float(arr.mean()) if arr.size else 0.0
    return np.zeros_like(arr) if mean <= 1e-9 else arr / mean

def clip_ph(x, config):
    return float(np.clip(x, config.ph_min, config.ph_max))

def clip_ec(x):
    return float(max(0.0, x))

def safe_features(d):
    out = {}
    for k, v in d.items():
        try:
            v = float(v)
        except Exception:
            v = 0.0
        out[k] = v if math.isfinite(v) else 0.0
    return out

def to_builtin(v):
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    if isinstance(v, pd.Timestamp):
        return v.isoformat()
    if isinstance(v, dict):
        return {k: to_builtin(x) for k, x in v.items()}
    if isinstance(v, list):
        return [to_builtin(x) for x in v]
    return v


def load_master(path, step_minutes):
    df = pd.read_csv(path, parse_dates=["timestamp"]).sort_values("timestamp").set_index("timestamp")
    df.index = pd.to_datetime(df.index)
    df = df[~df.index.duplicated(keep="last")]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    expected = pd.date_range(df.index.min(), df.index.max(), freq=f"{step_minutes}min")
    both = df["ph"].notna() & df["ec_ms"].notna()
    info = {
        "rows": int(len(df)),
        "joint_labeled_samples": int(both.sum()),
        "expected_grid_rows": int(len(expected)),
        "missing_grid_rows": int(len(expected.difference(df.index))),
    }
    return df, info

def build_segments(df):
    times = list(df.index[df["ph"].notna() & df["ec_ms"].notna()])
    out = []
    for i in range(len(times) - 1):
        t0, t1 = times[i], times[i + 1]
        gap_h = float((t1 - t0).total_seconds() / 3600.0)
        if gap_h <= 0:
            continue
        out.append(Segment(i, t0, t1, gap_h, float(df.loc[t0, "ph"]), float(df.loc[t1, "ph"]), float(df.loc[t0, "ec_ms"]), float(df.loc[t1, "ec_ms"])))
    return out

def history_features(df, current_time):
    recent = slice_df(df, current_time - pd.Timedelta(hours=6), current_time)
    mid = slice_df(df, current_time - pd.Timedelta(hours=12), current_time - pd.Timedelta(hours=6))
    prior = slice_df(df, current_time - pd.Timedelta(hours=24), current_time - pd.Timedelta(hours=12))
    full = slice_df(df, current_time - pd.Timedelta(hours=24), current_time)
    hist_salt_ramp = 0.0
    hist_acid_decay = 0.0
    if len(full):
        salt_d = sum_series(full, SALT_FERTS)
        acid_d = sum_series(full, ACID_FERTS)
        if (salt_d > 0).any():
            hrs = np.asarray((current_time - full.index[salt_d > 0]).total_seconds()) / 3600.0
            hist_salt_ramp = float((salt_d[salt_d > 0].to_numpy() * (1.0 - np.exp(-0.34 * hrs))).sum())
        if (acid_d > 0).any():
            hrs = np.asarray((current_time - full.index[acid_d > 0]).total_seconds()) / 3600.0
            hist_acid_decay = float((acid_d[acid_d > 0].to_numpy() * np.exp(-0.34 * hrs)).sum())
    irr = to_num(full["irrigation_ml_current"]) if "irrigation_ml_current" in full.columns else pd.Series(0.0, index=full.index)
    return {
        "hist_irr_recent": float(irr.loc[recent.index].sum()) if len(recent) else 0.0,
        "hist_irr_mid": float(irr.loc[mid.index].sum()) if len(mid) else 0.0,
        "hist_irr_prior": float(irr.loc[prior.index].sum()) if len(prior) else 0.0,
        "hist_salt_ramp": hist_salt_ramp,
        "hist_acid_decay": hist_acid_decay,
        "hist_salt_buildup": float(sum_avail(full, SALT_FERTS) - float(irr.sum()) * 0.1) if len(full) else 0.0,
        "hist_hrs_since_fert": hrs_since_last_positive(fert_any(full), current_time, 24.0),
        "hist_hrs_since_irr": hrs_since_last_positive(irr, current_time, 24.0),
    }

def recent_features(df, current_time, step_minutes):
    recent = slice_df(df, current_time - pd.Timedelta(hours=6), current_time)
    if len(recent) == 0:
        return {"recent_et0_sum_6h": 0.0, "recent_temp_mean_6h": 0.0, "recent_rad_integral_6h": 0.0, "recent_vpd_mean_6h": 0.0}
    temp = to_num(recent["internal_air_temp_c"]) if "internal_air_temp_c" in recent.columns else pd.Series(0.0, index=recent.index)
    rh = to_num(recent["internal_rh_%"], 100.0) if "internal_rh_%" in recent.columns else pd.Series(100.0, index=recent.index)
    rad = to_num(recent["internal_radiation"]) if "internal_radiation" in recent.columns else pd.Series(0.0, index=recent.index)
    et0 = to_num(recent["ET0"]) if "ET0" in recent.columns else pd.Series(0.0, index=recent.index)
    return {
        "recent_et0_sum_6h": float(et0.sum()),
        "recent_temp_mean_6h": float(temp.mean()),
        "recent_rad_integral_6h": float(rad.sum() * (step_minutes / 60.0)),
        "recent_vpd_mean_6h": float(np.mean(vpd(temp.to_numpy(), rh.to_numpy()))),
    }

def step_features(df, current_time, next_time, ph_now, ec_now, anchor_time, step_minutes):
    step = slice_df(df, current_time, next_time)
    acid = sum_avail(step, ACID_FERTS)
    salt = sum_avail(step, SALT_FERTS)
    fert = acid + salt
    irr = float(to_num(step["irrigation_ml_current"]).sum()) if "irrigation_ml_current" in step.columns else 0.0
    et0 = float(to_num(step["ET0"]).sum()) if "ET0" in step.columns else 0.0
    temp = float(to_num(step["internal_air_temp_c"]).mean()) if "internal_air_temp_c" in step.columns and len(step) else 0.0
    rh = float(to_num(step["internal_rh_%"], 100.0).mean()) if "internal_rh_%" in step.columns and len(step) else 100.0
    soil = float(to_num(step["soil_temp_pred"]).mean()) if "soil_temp_pred" in step.columns and len(step) else 0.0
    rad = float(to_num(step["internal_radiation"]).mean()) if "internal_radiation" in step.columns and len(step) else 0.0
    canopy = float(to_num(step["canopy_cover"]).mean()) if "canopy_cover" in step.columns and len(step) else 0.0
    vpd_step = float(vpd(np.array([temp]), np.array([rh]))[0])
    hour = current_time.hour + current_time.minute / 60.0
    feats = {
        "ph_current": ph_now,
        "ec_current": ec_now,
        "step_hours": float((next_time - current_time).total_seconds() / 3600.0),
        "hours_since_anchor": float(max(0.0, (current_time - anchor_time).total_seconds() / 3600.0)),
        "irrigation_step": irr,
        "fert_total_step": fert,
        "acid_step": acid,
        "salt_step": salt,
        "ET0_step": et0,
        "temp_step": temp,
        "soil_temp_step": soil,
        "radiation_step": rad,
        "canopy_step": canopy,
        "fert_conc_step": float(fert / (irr + 1.0)),
        "acid_conc_step": float(acid / (irr + 1.0)),
        "salt_balance_step": float(salt - irr),
        "vpd_step": vpd_step,
        "transpiration_step": float(vpd_step * canopy),
        "evapo_step": float(et0 * canopy),
        "log_ec_drive_step": float(np.log((fert / (irr + 1.0)) + 0.01) - np.log(max(ec_now, 0.0) + 0.01)),
        "hour_sin": float(np.sin(2 * np.pi * hour / 24.0)),
        "hour_cos": float(np.cos(2 * np.pi * hour / 24.0)),
    }
    feats.update(history_features(df, current_time))
    feats.update(recent_features(df, current_time, step_minutes))
    return safe_features(feats)


def segment_weights(df, segment, target, config):
    step = df.loc[(df.index >= segment.start_time) & (df.index < segment.end_time)]
    n = len(step)
    if n == 0:
        return np.array([], dtype=float)
    if config.pseudo_label_mode == "uniform" or n == 1:
        return np.full(n, 1.0 / n, dtype=float)
    irr = np.log1p(np.clip(to_num(step["irrigation_ml_current"]).to_numpy(), 0.0, None)) if "irrigation_ml_current" in step.columns else np.zeros(n)
    acid = np.log1p(np.clip(sum_series(step, ACID_FERTS).to_numpy(), 0.0, None))
    salt = np.log1p(np.clip(sum_series(step, SALT_FERTS).to_numpy(), 0.0, None))
    et0 = np.log1p(np.clip(to_num(step["ET0"]).to_numpy(), 0.0, None)) if "ET0" in step.columns else np.zeros(n)
    temp = to_num(step["internal_air_temp_c"]).to_numpy() if "internal_air_temp_c" in step.columns else np.zeros(n)
    rh = to_num(step["internal_rh_%"], 100.0).to_numpy() if "internal_rh_%" in step.columns else np.full(n, 100.0)
    soil = np.log1p(np.clip(to_num(step["soil_temp_pred"]).to_numpy(), 0.0, None)) if "soil_temp_pred" in step.columns else np.zeros(n)
    rad = np.log1p(np.clip(to_num(step["internal_radiation"]).to_numpy(), 0.0, None)) if "internal_radiation" in step.columns else np.zeros(n)
    canopy = np.clip(to_num(step["canopy_cover"]).to_numpy(), 0.0, None) if "canopy_cover" in step.columns else np.zeros(n)
    transp = np.log1p(np.clip(vpd(temp, rh) * canopy, 0.0, None))
    evapo = np.log1p(np.clip(np.exp(et0) - 1.0, 0.0, None) * np.maximum(canopy, 0.05))
    if target == "ph":
        driven = 1.0 + 0.30 * normalize(acid) + 0.20 * normalize(evapo) + 0.15 * normalize(transp) + 0.10 * normalize(irr) + 0.10 * normalize(rad) + 0.05 * normalize(soil)
    else:
        driven = 1.0 + 0.30 * normalize(salt) + 0.25 * normalize(irr) + 0.20 * normalize(evapo) + 0.15 * normalize(transp) + 0.10 * normalize(et0)
    driven = np.nan_to_num(driven, nan=0.0, posinf=0.0, neginf=0.0)
    uniform = np.full(n, 1.0 / n, dtype=float)
    if driven.sum() <= 1e-12:
        return uniform
    driven = driven / driven.sum()
    blend = float(np.clip(config.pseudo_uniform_blend, 0.0, 1.0))
    out = blend * uniform + (1.0 - blend) * driven
    return out / out.sum()


def pseudo_rows(df, segment, config):
    if segment.gap_hours > config.max_training_gap_hours:
        return pd.DataFrame()
    grid = df.loc[(df.index >= segment.start_time) & (df.index <= segment.end_time)].index
    if len(grid) < 2:
        return pd.DataFrame()
    ph_w = segment_weights(df, segment, "ph", config)
    ec_w = segment_weights(df, segment, "ec", config)
    n = len(grid) - 1
    if len(ph_w) != n or len(ec_w) != n:
        return pd.DataFrame()
    gap_weight = 1.0 if segment.gap_hours <= config.long_gap_downweight_hours else float(config.long_gap_downweight_hours / segment.gap_hours)
    ref_steps = max(1, int((config.reference_segment_hours * 60.0) / config.step_minutes))
    sample_weight = float(gap_weight * ref_steps / max(n, 1))
    ph_state, ec_state = segment.ph_start, segment.ec_start
    total_dph = segment.ph_end - segment.ph_start
    total_dec = segment.ec_end - segment.ec_start
    rows = []
    for i, (cur, nxt) in enumerate(zip(grid[:-1], grid[1:])):
        dph = float(total_dph * ph_w[i])
        dec = float(total_dec * ec_w[i])
        feats = step_features(df, cur, nxt, ph_state, ec_state, segment.start_time, config.step_minutes)
        feats.update({"timestamp": cur, "segment_id": segment.segment_id, "delta_ph_10m": dph, "delta_ec_10m": dec, "sample_weight": sample_weight})
        rows.append(feats)
        ph_state = clip_ph(ph_state + dph, config)
        ec_state = clip_ec(ec_state + dec)
    return pd.DataFrame(rows)


def matrix(df, cols):
    return df.reindex(columns=cols, fill_value=0.0).replace([np.inf, -np.inf], np.nan).fillna(0.0)


def build_model(target, family):
    family = family.lower()
    if family in {"auto", "xgb", "xgboost"} and xgb is not None:
        return xgb.XGBRegressor(**XGB_PARAMS[target]), "xgboost"
    if family in {"auto", "rf", "random_forest"}:
        return RandomForestRegressor(**RF_PARAMS[target]), "random_forest"
    raise ValueError(f"Unsupported model family: {family}")


def fit_model(train_df, target, family):
    cols = PH_FEATURES if target == "ph" else EC_FEATURES
    label = "delta_ph_10m" if target == "ph" else "delta_ec_10m"
    model, name = build_model(target, family)
    model.fit(matrix(train_df, cols), pd.to_numeric(train_df[label], errors="coerce").fillna(0.0).to_numpy(), sample_weight=pd.to_numeric(train_df["sample_weight"], errors="coerce").fillna(1.0).to_numpy())
    return model, name


def predict_delta(model, cols, feats):
    return float(model.predict(matrix(pd.DataFrame([feats]), cols))[0])


def feature_importance(model, cols):
    imp = getattr(model, "feature_importances_", None)
    if imp is None:
        return pd.DataFrame(columns=["feature", "importance"])
    return pd.DataFrame({"feature": cols, "importance": imp}).sort_values("importance", ascending=False).reset_index(drop=True)


def mae(y, yhat):
    return float(np.mean(np.abs(y - yhat))) if len(y) else float("nan")

def rmse(y, yhat):
    return float(np.sqrt(np.mean((y - yhat) ** 2))) if len(y) else float("nan")

def r2(y, yhat):
    if len(y) < 2:
        return float("nan")
    denom = float(np.sum((y - np.mean(y)) ** 2))
    return float("nan") if denom <= 1e-12 else float(1.0 - np.sum((y - yhat) ** 2) / denom)


def metrics_and_bins(endpoint_df):
    metrics = {"n_test_segments": int(len(endpoint_df))}
    if endpoint_df.empty:
        return metrics, pd.DataFrame()
    ph_true = endpoint_df["ph_true"].to_numpy(dtype=float)
    ph_pred = endpoint_df["ph_pred"].to_numpy(dtype=float)
    ph_naive = endpoint_df["ph_naive"].to_numpy(dtype=float)
    ec_true = endpoint_df["ec_true"].to_numpy(dtype=float)
    ec_pred = endpoint_df["ec_pred"].to_numpy(dtype=float)
    ec_naive = endpoint_df["ec_naive"].to_numpy(dtype=float)
    metrics.update({
        "ph_mae": mae(ph_true, ph_pred), "ph_rmse": rmse(ph_true, ph_pred), "ph_r2": r2(ph_true, ph_pred),
        "ph_mae_naive": mae(ph_true, ph_naive), "ph_gain_mae": mae(ph_true, ph_naive) - mae(ph_true, ph_pred),
        "ph_bias": float(np.mean(ph_pred - ph_true)),
        "ec_mae": mae(ec_true, ec_pred), "ec_rmse": rmse(ec_true, ec_pred), "ec_r2": r2(ec_true, ec_pred),
        "ec_mae_naive": mae(ec_true, ec_naive), "ec_gain_mae": mae(ec_true, ec_naive) - mae(ec_true, ec_pred),
        "ec_bias": float(np.mean(ec_pred - ec_true)),
    })
    temp = endpoint_df.copy()
    temp["gap_bin"] = pd.cut(temp["gap_hours"], bins=[0.0, 12.0, 24.0, np.inf], labels=["0-12h", "12-24h", "24h+"], include_lowest=True)
    rows = []
    for gap_bin, group in temp.groupby("gap_bin", observed=False):
        if len(group) == 0:
            continue
        rows.append({
            "gap_bin": str(gap_bin), "n": int(len(group)),
            "ph_mae": mae(group["ph_true"].to_numpy(dtype=float), group["ph_pred"].to_numpy(dtype=float)),
            "ph_naive_mae": mae(group["ph_true"].to_numpy(dtype=float), group["ph_naive"].to_numpy(dtype=float)),
            "ph_bias": float(np.mean(group["ph_pred"].to_numpy(dtype=float) - group["ph_true"].to_numpy(dtype=float))),
            "ec_mae": mae(group["ec_true"].to_numpy(dtype=float), group["ec_pred"].to_numpy(dtype=float)),
            "ec_naive_mae": mae(group["ec_true"].to_numpy(dtype=float), group["ec_naive"].to_numpy(dtype=float)),
            "ec_bias": float(np.mean(group["ec_pred"].to_numpy(dtype=float) - group["ec_true"].to_numpy(dtype=float))),
        })
    return metrics, pd.DataFrame(rows)



GAP_BIN_LABELS = ("0-12h", "12-24h", "24h+")


def gap_bin_label(gap_hours):
    if gap_hours <= 12.0:
        return "0-12h"
    if gap_hours <= 24.0:
        return "12-24h"
    return "24h+"


def build_stability_profile(train_df):
    ph_abs = np.abs(pd.to_numeric(train_df["delta_ph_10m"], errors="coerce").fillna(0.0).to_numpy())
    ec_abs = np.abs(pd.to_numeric(train_df["delta_ec_10m"], errors="coerce").fillna(0.0).to_numpy())
    ph_clip = float(np.clip(np.quantile(ph_abs, 0.99) if len(ph_abs) else 0.08, 0.04, 0.18))
    ec_clip = float(np.clip(np.quantile(ec_abs, 0.98) if len(ec_abs) else 0.05, 0.02, 0.16))
    return {"ph_clip": ph_clip, "ec_clip": ec_clip}


def horizon_shrink(hours_since_anchor, start_hours, min_scale):
    if hours_since_anchor <= start_hours:
        return 1.0
    return float(max(min_scale, start_hours / max(hours_since_anchor, start_hours)))


def anchor_weight(hours_since_anchor, start_hours, max_weight):
    if hours_since_anchor <= start_hours:
        return 0.0
    excess = hours_since_anchor - start_hours
    return float(min(max_weight, max_weight * (excess / max(start_hours, 1e-6))))


def ec_signal_trust(features, hours_since_anchor):
    signal = (
        np.log1p(max(features.get("salt_step", 0.0), 0.0))
        + 0.7 * np.log1p(abs(features.get("salt_balance_step", 0.0)))
        + 0.4 * np.log1p(max(features.get("fert_total_step", 0.0), 0.0))
        + 0.3 * np.log1p(max(features.get("irrigation_step", 0.0), 0.0))
        + 0.2 * np.log1p(max(features.get("hist_salt_ramp", 0.0), 0.0))
    )
    base_trust = float(np.clip(signal / 4.0, 0.18, 0.92))
    long_gap_penalty = horizon_shrink(hours_since_anchor, 8.0, 0.45)
    return float(np.clip(base_trust * long_gap_penalty, 0.15, 0.85))


def init_gap_calibration():
    return {target: {label: [] for label in GAP_BIN_LABELS} for target in ("ph", "ec")}


def gap_bias(calibration_state, target, gap_hours):
    if not calibration_state:
        return 0.0
    bucket = calibration_state[target][gap_bin_label(gap_hours)]
    if not bucket:
        return 0.0
    return float(np.median(np.asarray(bucket[-24:], dtype=float)))


def calibration_weight(target, gap_hours):
    if target == "ph":
        if gap_hours <= 12.0:
            return 0.0
        if gap_hours <= 24.0:
            return 1.0
        return 0.85
    if gap_hours <= 24.0:
        return 0.0
    return 0.5


def calibrated_bias(calibration_state, target, gap_hours):
    weight = calibration_weight(target, gap_hours)
    if weight <= 0.0:
        return 0.0
    raw_bias = gap_bias(calibration_state, target, gap_hours)
    cap = 1.2 if target == "ph" else 0.18
    return float(np.clip(weight * raw_bias, -cap, cap))


def update_gap_calibration(calibration_state, endpoint_row, max_history=24):
    for target in ("ph", "ec"):
        bucket = calibration_state[target][gap_bin_label(float(endpoint_row["gap_hours"]))]
        bucket.append(float(endpoint_row[f"{target}_true"] - endpoint_row[f"{target}_pred_raw"]))
        if len(bucket) > max_history:
            del bucket[:-max_history]


def calibration_summary(calibration_state):
    rows = []
    representative_gap_hours = {"0-12h": 6.0, "12-24h": 18.0, "24h+": 36.0}
    for target in ("ph", "ec"):
        for gap_bin in GAP_BIN_LABELS:
            values = calibration_state[target][gap_bin]
            rows.append({
                "target": target,
                "gap_bin": gap_bin,
                "n_residuals": int(len(values)),
                "median_bias": float(np.median(values)) if values else 0.0,
                "mean_bias": float(np.mean(values)) if values else 0.0,
                "applied_weight": float(calibration_weight(target, representative_gap_hours[gap_bin])),
            })
    return pd.DataFrame(rows)


def apply_gap_calibration(dense_rows, segment, calibration_state, config):
    ph_bias = calibrated_bias(calibration_state, "ph", segment.gap_hours)
    ec_bias = calibrated_bias(calibration_state, "ec", segment.gap_hours)
    total_gap = max(float(segment.gap_hours), 1e-9)
    for row in dense_rows:
        row["ph_pred_raw"] = row.get("ph_pred_raw", row["ph_pred"])
        row["ec_pred_raw"] = row.get("ec_pred_raw", row["ec_pred"])
        progress = min(1.0, max(0.0, float(row.get("hours_since_anchor", 0.0)) / total_gap))
        row["ph_pred"] = clip_ph(row["ph_pred_raw"] + progress * ph_bias, config)
        row["ec_pred"] = clip_ec(row["ec_pred_raw"] + progress * ec_bias)
    return dense_rows


def bootstrap_gap_calibration(df, warmup_segments, ph_model, ec_model, config, stability_profile):
    calibration_state = init_gap_calibration()
    seed_segments = warmup_segments[-min(24, len(warmup_segments)):]
    for segment in seed_segments:
        _, endpoint = rollout_segment(
            df,
            segment,
            ph_model,
            ec_model,
            config,
            stability_profile=stability_profile,
            calibration_state=None,
            apply_calibration=False,
        )
        for target in ("ph", "ec"):
            calibration_state[target][gap_bin_label(segment.gap_hours)].append(
                float(endpoint[f"{target}_true"] - endpoint[f"{target}_pred_raw"])
            )
    return calibration_state


def rollout_segment(df, segment, ph_model, ec_model, config, stability_profile=None, calibration_state=None, apply_calibration=True):
    profile = stability_profile or {"ph_clip": 0.08, "ec_clip": 0.05}
    grid = df.loc[(df.index >= segment.start_time) & (df.index <= segment.end_time)].index
    if len(grid) < 2:
        endpoint = {
            "segment_id": segment.segment_id,
            "start_time": segment.start_time,
            "end_time": segment.end_time,
            "gap_hours": segment.gap_hours,
            "ph_pred_raw": segment.ph_start,
            "ph_pred": segment.ph_start,
            "ph_true": segment.ph_end,
            "ph_naive": segment.ph_start,
            "ec_pred_raw": segment.ec_start,
            "ec_pred": segment.ec_start,
            "ec_true": segment.ec_end,
            "ec_naive": segment.ec_start,
        }
        return [], endpoint

    ph_state, ec_state = segment.ph_start, segment.ec_start
    dense = [{
        "segment_id": segment.segment_id,
        "anchor_time": segment.start_time,
        "timestamp": segment.start_time,
        "hours_since_anchor": 0.0,
        "ph_pred_raw": segment.ph_start,
        "ph_pred": segment.ph_start,
        "ec_pred_raw": segment.ec_start,
        "ec_pred": segment.ec_start,
        "delta_ph_pred": 0.0,
        "delta_ec_pred": 0.0,
        "ph_true": segment.ph_start,
        "ec_true": segment.ec_start,
        "is_real_sample": True,
        "is_endpoint": False,
    }]

    for cur, nxt in zip(grid[:-1], grid[1:]):
        hours_since_anchor = float((nxt - segment.start_time).total_seconds() / 3600.0)
        feats = step_features(df, cur, nxt, ph_state, ec_state, segment.start_time, config.step_minutes)

        dph_raw = predict_delta(ph_model, PH_FEATURES, feats)
        dph_raw = float(np.clip(dph_raw, -profile["ph_clip"], profile["ph_clip"]))
        dph = dph_raw * horizon_shrink(hours_since_anchor, 6.0, 0.30)
        ph_candidate = clip_ph(ph_state + dph, config)
        ph_pull = anchor_weight(hours_since_anchor, 6.0, 0.025)
        ph_state = clip_ph((1.0 - ph_pull) * ph_candidate + ph_pull * segment.ph_start, config)

        dec_raw = predict_delta(ec_model, EC_FEATURES, feats)
        dec_raw = float(np.clip(dec_raw, -profile["ec_clip"], profile["ec_clip"]))
        dec = dec_raw * horizon_shrink(hours_since_anchor, 4.0, 0.25)
        ec_candidate = clip_ec(ec_state + dec)
        trust = ec_signal_trust(feats, hours_since_anchor)
        ec_state = clip_ec((trust * ec_candidate) + ((1.0 - trust) * segment.ec_start))

        dense.append({
            "segment_id": segment.segment_id,
            "anchor_time": segment.start_time,
            "timestamp": nxt,
            "hours_since_anchor": hours_since_anchor,
            "ph_pred_raw": ph_state,
            "ph_pred": ph_state,
            "ec_pred_raw": ec_state,
            "ec_pred": ec_state,
            "delta_ph_pred": dph,
            "delta_ec_pred": dec,
            "ph_true": segment.ph_end if nxt == segment.end_time else np.nan,
            "ec_true": segment.ec_end if nxt == segment.end_time else np.nan,
            "is_real_sample": bool(nxt == segment.end_time),
            "is_endpoint": bool(nxt == segment.end_time),
        })

    if apply_calibration and calibration_state is not None:
        dense = apply_gap_calibration(dense, segment, calibration_state, config)

    last = dense[-1]
    endpoint = {
        "segment_id": segment.segment_id,
        "start_time": segment.start_time,
        "end_time": segment.end_time,
        "gap_hours": segment.gap_hours,
        "ph_pred_raw": last.get("ph_pred_raw", last["ph_pred"]),
        "ph_pred": last["ph_pred"],
        "ph_true": segment.ph_end,
        "ph_naive": segment.ph_start,
        "ec_pred_raw": last.get("ec_pred_raw", last["ec_pred"]),
        "ec_pred": last["ec_pred"],
        "ec_true": segment.ec_end,
        "ec_naive": segment.ec_start,
    }
    return dense, endpoint

def run_walkforward(df, segments, config):
    if len(segments) <= config.warmup_segments:
        raise ValueError(f"Need more than {config.warmup_segments} segments, found {len(segments)}.")
    train_parts = [pseudo_rows(df, s, config) for s in segments[:config.warmup_segments]]
    train_parts = [x for x in train_parts if not x.empty]
    if not train_parts:
        raise ValueError("No trainable pseudo-step rows were created from the warmup segments.")

    train_df = pd.concat(train_parts, ignore_index=True)
    ph_model, ph_name = fit_model(train_df, "ph", config.model_family)
    ec_model, ec_name = fit_model(train_df, "ec", config.model_family)
    stability_profile = build_stability_profile(train_df)
    calibration_state = bootstrap_gap_calibration(df, segments[:config.warmup_segments], ph_model, ec_model, config, stability_profile)

    dense_rows, endpoint_rows, skipped = [], [], 0
    for segment in segments[config.warmup_segments:]:
        dense, endpoint = rollout_segment(
            df,
            segment,
            ph_model,
            ec_model,
            config,
            stability_profile=stability_profile,
            calibration_state=calibration_state,
            apply_calibration=True,
        )
        dense_rows.extend(dense)
        endpoint_rows.append(endpoint)
        update_gap_calibration(calibration_state, endpoint)

        new_rows = pseudo_rows(df, segment, config)
        if new_rows.empty:
            skipped += 1
            continue
        train_df = pd.concat([train_df, new_rows], ignore_index=True)
        ph_model, ph_name = fit_model(train_df, "ph", config.model_family)
        ec_model, ec_name = fit_model(train_df, "ec", config.model_family)
        stability_profile = build_stability_profile(train_df)

    endpoint_df = pd.DataFrame(endpoint_rows).sort_values("end_time").reset_index(drop=True)
    dense_df = pd.DataFrame(dense_rows).sort_values(["segment_id", "timestamp"]).reset_index(drop=True)
    metrics, gap_bin_df = metrics_and_bins(endpoint_df)
    metrics.update({
        "warmup_segments": config.warmup_segments,
        "train_step_rows": int(len(train_df)),
        "segments_total": int(len(segments)),
        "segments_evaluated": int(len(endpoint_df)),
        "segments_skipped_for_pseudo_training": int(skipped),
        "ph_model_family": ph_name,
        "ec_model_family": ec_name,
        "ph_step_clip": float(stability_profile["ph_clip"]),
        "ec_step_clip": float(stability_profile["ec_clip"]),
    })
    return {
        "dense_df": dense_df,
        "endpoint_df": endpoint_df,
        "gap_bin_df": gap_bin_df,
        "metrics": metrics,
        "ph_importance": feature_importance(ph_model, PH_FEATURES),
        "ec_importance": feature_importance(ec_model, EC_FEATURES),
        "calibration_df": calibration_summary(calibration_state),
    }


def save_outputs(output_dir, config, dataset_info, results):
    output_dir.mkdir(parents=True, exist_ok=True)
    results["dense_df"].to_csv(output_dir / "dense_rollout_predictions.csv", index=False)
    results["endpoint_df"].to_csv(output_dir / "segment_endpoint_eval.csv", index=False)
    results["gap_bin_df"].to_csv(output_dir / "gap_bin_metrics.csv", index=False)
    if not results["ph_importance"].empty:
        results["ph_importance"].to_csv(output_dir / "ph_feature_importance.csv", index=False)
    if not results["ec_importance"].empty:
        results["ec_importance"].to_csv(output_dir / "ec_feature_importance.csv", index=False)
    if "calibration_df" in results and not results["calibration_df"].empty:
        results["calibration_df"].to_csv(output_dir / "calibration_by_gap.csv", index=False)
    (output_dir / "run_config.json").write_text(json.dumps(to_builtin(asdict(config)), indent=2), encoding="utf-8")
    (output_dir / "dataset_info.json").write_text(json.dumps(to_builtin(dataset_info), indent=2), encoding="utf-8")
    (output_dir / "metrics.json").write_text(json.dumps(to_builtin(results["metrics"]), indent=2), encoding="utf-8")


def parse_args():
    p = argparse.ArgumentParser(description="Recursive 10-minute root-zone EC/pH forecasting.")
    p.add_argument("--input", default="data/processed/master.csv")
    p.add_argument("--output-dir", default="data/processed/recursive_10min")
    p.add_argument("--step-minutes", type=int, default=10)
    p.add_argument("--warmup-segments", type=int, default=50)
    p.add_argument("--model-family", default="auto", choices=["auto", "xgboost", "xgb", "random_forest", "rf"])
    p.add_argument("--pseudo-label-mode", default="driver_blend", choices=["driver_blend", "uniform"])
    p.add_argument("--pseudo-uniform-blend", type=float, default=0.60)
    p.add_argument("--max-training-gap-hours", type=float, default=72.0)
    p.add_argument("--long-gap-downweight-hours", type=float, default=24.0)
    p.add_argument("--reference-segment-hours", type=float, default=6.0)
    return p.parse_args()


def main():
    args = parse_args()
    config = RunConfig(
        input_path=args.input,
        output_dir=args.output_dir,
        step_minutes=args.step_minutes,
        warmup_segments=args.warmup_segments,
        model_family=args.model_family,
        pseudo_label_mode=args.pseudo_label_mode,
        pseudo_uniform_blend=args.pseudo_uniform_blend,
        max_training_gap_hours=args.max_training_gap_hours,
        long_gap_downweight_hours=args.long_gap_downweight_hours,
        reference_segment_hours=args.reference_segment_hours,
    )
    df, info = load_master(Path(config.input_path), config.step_minutes)
    segments = build_segments(df)
    info["segments"] = int(len(segments))
    results = run_walkforward(df, segments, config)
    save_outputs(Path(config.output_dir), config, info, results)
    m = results["metrics"]
    print(f"Rows: {info['rows']}")
    print(f"Joint labeled samples: {info['joint_labeled_samples']}")
    print(f"Segments: {info['segments']}")
    print(f"pH  MAE={m.get('ph_mae', float('nan')):.4f}  naive={m.get('ph_mae_naive', float('nan')):.4f}")
    print(f"EC  MAE={m.get('ec_mae', float('nan')):.4f}  naive={m.get('ec_mae_naive', float('nan')):.4f}")
    print(f"Outputs written to: {config.output_dir}")



In [ ]:
config = RunConfig(`n    input_path="data/processed/master.csv",`n    output_dir="data/processed/recursive_10min",`n    step_minutes=10,`n    warmup_segments=50,`n    model_family="auto",`n    pseudo_label_mode="driver_blend",`n    pseudo_uniform_blend=0.60,`n    max_training_gap_hours=72.0,`n    long_gap_downweight_hours=24.0,`n    reference_segment_hours=6.0,`n)`nconfig`n

In [ ]:
df, info = load_master(Path(config.input_path), config.step_minutes)`nsegments = build_segments(df)`ninfo["segments"] = int(len(segments))`nresults = run_walkforward(df, segments, config)`nsave_outputs(Path(config.output_dir), config, info, results)`nresults["metrics"]`n

In [ ]:
results["gap_bin_df"]`n